In [1]:
# Import necessary libraries
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import os

In [2]:
# Set parameters
DATA_DIR = 'C:/Users/Angelina/Desktop/Waste-Classification/data/garbage_classification'  # path to your dataset folder
MODELS_DIR = 'C:/Users/Angelina/Desktop/Waste-Classification/models'  # path to save trained models
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)
EPOCHS = 25
LEARNING_RATE = 0.0001

In [3]:
# Create data generators with augmentation for training and simple rescaling for validation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,  # 20% for validation
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    fill_mode='nearest'
)

train_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=42
)

validation_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=42
)

Found 12415 images belonging to 12 classes.
Found 3100 images belonging to 12 classes.


In [4]:
# Load pre-trained DenseNet121 without top layer and freeze the base model initially
base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=IMAGE_SIZE + (3,))
base_model.trainable = False

In [5]:
# Add classification head
x = base_model.output
x = GlobalAveragePooling2D()(x)
output_layer = Dense(train_generator.num_classes, activation='softmax')(x)

In [6]:
# Define the model
model = Model(inputs=base_model.input, outputs=output_layer)

# Compile the model
model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Show model summary
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d      │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,408 │ zero_padding2d[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_1    │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 56, 56,    │          0 │ zero_padding2d_1… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 56, 56,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 56, 56,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 56, 56,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_relu │ (None, 56, 56,    │          0 │ conv2_block2_0_b… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_1_conv │ (None, 56, 56,    │     12,288 │ conv2_block2_0_r

 Total params: 7,049,804 (26.89 MB)

 Trainable params: 12,300 (48.05 KB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [7]:
# Train the model (first stage: train top layers)
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=EPOCHS,
    verbose=1
)

Epoch 1/25
388/388 ━━━━━━━━━━━━━━━━━━━━ 2809s 7s/step - accuracy: 0.4930 - loss: 1.6697 - val_accuracy: 0.6394 - val_loss: 1.2043
Epoch 2/25
388/388 ━━━━━━━━━━━━━━━━━━━━ 33668s 87s/step - accuracy: 0.7489 - loss: 0.9272 - val_accuracy: 0.7390 - val_loss: 0.8668
Epoch 3/25
388/388 ━━━━━━━━━━━━━━━━━━━━ 1103s 3s/step - accuracy: 0.8248 - loss: 0.6748 - val_accuracy: 0.7923 - val_loss: 0.7039
Epoch 4/25
388/388 ━━━━━━━━━━━━━━━━━━━━ 1107s 3s/step - accuracy: 0.8540 - loss: 0.5514 - val_accuracy: 0.8161 - val_loss: 0.6096
Epoch 5/25
388/388 ━━━━━━━━━━━━━━━━━━━━ 27502s 71s/step - accuracy: 0.8701 - loss: 0.4803 - val_accuracy: 0.8326 - val_loss: 0.5456
Epoch 6/25
388/388 ━━━━━━━━━━━━━━━━━━━━ 1341s 3s/step - accuracy: 0.8842 - loss: 0.4269 - val_accuracy: 0.8426 - val_loss: 0.5107
Epoch 7/25
388/388 ━━━━━━━━━━━━━━━━━━━━ 1344s 3s/step - accuracy: 0.8917 - loss: 0.3925 - val_accuracy: 0.8552 - val_loss: 0.4672
Epoch 8/25
388/388 ━━━━━━━━━━━━━━━━━━━━ 1388s 4s/step - accuracy: 0.9025 - loss: 0.360

In [8]:
# Unfreeze some layers in base model for fine-tuning
base_model.trainable = True
# Optional: freeze some layers in base_model to prevent overfitting
for layer in base_model.layers[:300]:
    layer.trainable = False

In [9]:
# Recompile with lower learning rate
model.compile(optimizer=Adam(learning_rate=LEARNING_RATE / 10),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [10]:
# Continue training (fine-tuning)
fine_tune_epochs = 10
total_epochs = EPOCHS + fine_tune_epochs

history_fine = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=total_epochs,
    initial_epoch=history.epoch[-1],
    verbose=1
)

Epoch 25/35
388/388 ━━━━━━━━━━━━━━━━━━━━ 1668s 4s/step - accuracy: 0.9046 - loss: 0.3128 - val_accuracy: 0.9013 - val_loss: 0.3009
Epoch 26/35
388/388 ━━━━━━━━━━━━━━━━━━━━ 1495s 4s/step - accuracy: 0.9312 - loss: 0.2192 - val_accuracy: 0.9155 - val_loss: 0.2711
Epoch 27/35
388/388 ━━━━━━━━━━━━━━━━━━━━ 1591s 4s/step - accuracy: 0.9455 - loss: 0.1808 - val_accuracy: 0.9139 - val_loss: 0.2466
Epoch 28/35
388/388 ━━━━━━━━━━━━━━━━━━━━ 1642s 4s/step - accuracy: 0.9526 - loss: 0.1558 - val_accuracy: 0.9274 - val_loss: 0.2264
Epoch 29/35
388/388 ━━━━━━━━━━━━━━━━━━━━ 1646s 4s/step - accuracy: 0.9578 - loss: 0.1412 - val_accuracy: 0.9229 - val_loss: 0.2277
Epoch 30/35
388/388 ━━━━━━━━━━━━━━━━━━━━ 1634s 4s/step - accuracy: 0.9597 - loss: 0.1328 - val_accuracy: 0.9371 - val_loss: 0.2063
Epoch 31/35
388/388 ━━━━━━━━━━━━━━━━━━━━ 1635s 4s/step - accuracy: 0.9625 - loss: 0.1231 - val_accuracy: 0.9319 - val_loss: 0.2046
Epoch 32/35
388/388 ━━━━━━━━━━━━━━━━━━━━ 1637s 4s/step - accuracy: 0.9671 - loss: 0

In [16]:
# Save trained model
os.makedirs(MODELS_DIR, exist_ok=True)
model_save_path = os.path.join(MODELS_DIR, 'densenet121_waste_classifier.keras')
model.save(model_save_path)

print(f"Model saved to {model_save_path}")

Model saved to C:/Users/Angelina/Desktop/Waste-Classification/models\densenet121_waste_classifier.keras


In [14]:
class_indices = train_generator.class_indices
print(class_indices)

{'battery': 0, 'biological': 1, 'brown-glass': 2, 'cardboard': 3, 'clothes': 4, 'green-glass': 5, 'metal': 6, 'paper': 7, 'plastic': 8, 'shoes': 9, 'trash': 10, 'white-glass': 11}


In [15]:
# Sort class names by their index values
class_names = [k for k, v in sorted(class_indices.items(), key=lambda item: item[1])]
print(class_names)


['battery', 'biological', 'brown-glass', 'cardboard', 'clothes', 'green-glass', 'metal', 'paper', 'plastic', 'shoes', 'trash', 'white-glass']


In [21]:
model = tf.keras.models.load_model(model_save_path)

In [22]:
from tensorflow.keras.optimizers import RMSprop
model.compile(optimizer=RMSprop(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])